# AACAP Challenge 3
__Marco Cioci__ - 10694772, __Francesco Rosnati__ - 10562427, __Luca Tramacere__ - 10676881




# Critical aspects


### Exploit data locality (memory hierarchy)
It's always important to exploit data locality. Cuda allows it through the proper use of memory hierarchy.
- __Thread local memory__: during for loop each threads sums partial result in a local variable.
- __Shared memory__: previous result is then saved by each thread in a shared memory array. The array is then reduced to a single value for each block.  

### Vectorization
Cuda allows to use specif data types to perform vectorization (`uchar4` in this case), in order to be able to better exploit SIMD.


### Loop unrolling
Loop unrolling procedure suggests the compiler to reduce branching instructions when allowed. The unrolling factor is empirically established.
Each for loop of Cuda kernels tries to exploit it, every loop is set to a for loop in order to be able to.


### Reduction parallel pattern
For the required task it was completely avoidable the use of any atomic operation, in particular the vector reduction parallel pattern is a more performing choice. It is used to reduce the result obtained by each thread in a single value for each block. Again all those values are reduced to a single result by the second kernel.

### Monodimensional indexing of threads:
Due to semplicity of the required function it's not necessarily needed the use of bi-dimensional thread indexing.
In a problem like matrix-matrix multiplication you would need at least 2 indexes to guarantee coalesced memory accesses.
The solution applied here follows a simpler idea: _each thread loops over an entire row_ to perform the multiplication, memory accesses are already coalesced.


### Cuda Malloc Host:
The standard malloc function from the stdlib.h library allocates memory in a way that allows paging (virtual memory). This is not efficient for transferring data to GPU device memory because the device memory cannot directly access pages stored on an SSD. Each time the device memory requires data, the host RAM must first fetch it from the SSD, causing a delay. The cudaMallocHost function allocates __pinned memory__ on the host RAM, ensuring that the data is always ready and immediately available for transfer to the GPU device memory, thereby improving the efficiency of data transfers.

# CODE
The average execution time is around 29 milliseconds, but we have noticed that the initial runs are slightly slower.

__Please perform a few warm-up runs.__

In [ ]:
!nvcc --version
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Tue_Aug_15_22:02:13_PDT_2023
Cuda compilation tools, release 12.2, V12.2.140
Build cuda_12.2.r12.2/compiler.33191640_0
The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [ ]:
%%cuda
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda.h>

#define DIM_SIZE 16384


void printmatrix(unsigned char* a)
{
    printf("Matrix: \n");
    for(int i = 0; i < DIM_SIZE; i++)
    {
      for(int j = 0; j < DIM_SIZE; j++)
      {
          printf("%i ", a[i*DIM_SIZE + j]);
      }
      printf("\n");
    }
}

void printvector(unsigned char* a)
{
    printf("Vector: \n");
    for(int j = 0; j < DIM_SIZE; j++)
    {
        printf("%i ", a[j]);
    }
    printf("\n");
}

long int cpu_function (unsigned char *a, unsigned char *b)
{
    int row , column;
    long int * c;
    c = (long int*) malloc(sizeof(long int)*DIM_SIZE);
    for (row = 0; row < DIM_SIZE; row++)
    {
        long int temp = 0;
        for (column = 0; column < DIM_SIZE; column++)
        {
            temp += a[row*DIM_SIZE+column] * b[column] * b[column] * b[column];
        }
        c[row] = temp;
    }
    long int sum = 0;
    for (int i = 0; i < DIM_SIZE; i++)
    {
        sum += c[i];
    }
    free(c);
    return sum;
}

__global__ void mulAndReduction(unsigned char *A, unsigned char *b, long int *result)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;  // single dimension indexing

    __shared__ long int temp[1024];  // exploit block shared memory
    temp[threadIdx.x] = 0;

    long int sum = 0;  // exploit thread local memory
    if (i < DIM_SIZE) {
        #pragma unroll 8  // loop unrolling
        for (int k = 0; k < DIM_SIZE; k += 4)
        {
            // vectorization
            uchar4 A_vec = *(reinterpret_cast<uchar4 *>(&A[i * DIM_SIZE + k]));
            uchar4 b_vec = *(reinterpret_cast<uchar4 *>(&b[k]));

            sum += A_vec.x * b_vec.x * b_vec.x * b_vec.x;
            sum += A_vec.y * b_vec.y * b_vec.y * b_vec.y;
            sum += A_vec.z * b_vec.z * b_vec.z * b_vec.z;
            sum += A_vec.w * b_vec.w * b_vec.w * b_vec.w;
        }
        //  thread result to shared
        temp[threadIdx.x] = sum;
    }

    __syncthreads();  // barrier

    // vector reduction threads within a block
    #pragma unroll 8
    for (int stride = 1; stride < blockDim.x; stride *= 2)
    {
        int index = 2 * stride * threadIdx.x;
        if (index < blockDim.x)
        {
            temp[index] += temp[index + stride];
        }
        __syncthreads();
    }

    // each block returns a value
    if (threadIdx.x == 0)
    {
        result[blockIdx.x] = temp[0];
    }
}

__global__ void singleBlockReduction(long int *vec, int p)
{
    int thread = threadIdx.x;

    #pragma unroll 8
    for (int t = 1, k = p / 2; k >= 1 && thread < k; k /= 2, t *= 2)
    {
        vec[thread * 2 * t] = vec[thread * 2 * t] + vec[thread * 2 * t + t];
    }
}

int main(int argc, char const *argv[])
{
    // retrive some info about the CUDA device
    int nDevices;
    cudaGetDeviceCount(&nDevices);
    for (int i = 0; i < nDevices; i++)
    {
      cudaDeviceProp prop;
      cudaGetDeviceProperties(&prop, i);
      printf("Device Number: %d\n", i);
      printf("  Device name: %s\n", prop.name);
      printf("  max Blocks Per MultiProcessor: %d\n", prop.maxBlocksPerMultiProcessor);
      printf("  max Threads Per MultiProcessor: %d\n", prop.maxThreadsPerMultiProcessor);
      printf("  max Threads Per Block: %d\n", prop.maxThreadsPerBlock);
      printf("  num SM: %d\n", prop.multiProcessorCount);
      printf("  num bytes sharedMem Per Block: %d\n", prop.sharedMemPerBlock);
      printf("  num bytes sharedMem Per Multiprocessor: %d\n", prop.sharedMemPerMultiprocessor);
      printf("  Memory Clock Rate (KHz): %d\n",
           prop.memoryClockRate);
      printf("  Memory Bus Width (bits): %d\n",
           prop.memoryBusWidth);
      printf("  Peak Memory Bandwidth (GB/s): %f\n\n",
           2.0*prop.memoryClockRate*(prop.memoryBusWidth/8)/1.0e6);
    }

    srand(time(NULL));

    unsigned char * matr, * vect;
    long int cpu_result;
    long int gpu_result;

    // Data allocation - exploit pinned memory for best performance
    cudaMallocHost((void**)&matr, DIM_SIZE * DIM_SIZE * sizeof(unsigned char));
    cudaMallocHost((void**)&vect, DIM_SIZE * sizeof(unsigned char));

    unsigned char *d_matr;
    cudaMalloc((void**)&d_matr, DIM_SIZE * DIM_SIZE * sizeof(unsigned char));
    unsigned char *d_vect;
    cudaMalloc((void**)&d_vect, DIM_SIZE * sizeof(unsigned char));


    // Initialization
    clock_t init_begin = clock();

    // initialize matrix A
    for (int i = 0; i < DIM_SIZE; i++)
    {
        for (int j = 0; j < DIM_SIZE; j++)
        {
              matr[i * DIM_SIZE + j] = ((unsigned char)rand()) % 2;
        }
    }

    //printmatrix(matr);

    // initialize vector B
    for (int j = 0; j < DIM_SIZE; j++)
    {
      vect[j] = ((unsigned char)rand()) % 3;
    }

    //printvector(vect);

    clock_t init_end = clock();
    double init_time = ((double)((init_end - init_begin)) * 1000) / CLOCKS_PER_SEC;
    printf("Time elapsed on initialization with size %d = %f ms\n\n", DIM_SIZE, init_time);


    // sequential version of matrix multiplication
    clock_t begin = clock();
    cpu_result = cpu_function(matr, vect);
    clock_t end = clock();
    double time_spent = ((double)((end - begin)) * 1000) / CLOCKS_PER_SEC;
    printf("Time elapsed on naive CPU implementation with size %d = %f ms\n\n", DIM_SIZE, time_spent);


    float  gpu_elapsed_time_ms;

    // some events to count the execution time
    //clock_t st, end;
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start, 0);


    // gpu computation

    cudaMemcpy(d_matr, matr, DIM_SIZE * DIM_SIZE * sizeof(unsigned char), cudaMemcpyHostToDevice);
    cudaMemcpy(d_vect, vect, DIM_SIZE * sizeof(unsigned char), cudaMemcpyHostToDevice);

    int THREADS = 64;
    int blocks = (DIM_SIZE  + THREADS - 1) / THREADS;
    //printf("\nNumber of blocks: %d\n", blocks);

    long int *d_gpu_result;
    cudaMalloc(&d_gpu_result, blocks * sizeof(long int));
    cudaMemset(d_gpu_result, 0, blocks * sizeof(long int));

    // performs A[j][k] * b[k]^3, reduce all elements within a block
    mulAndReduction<<<blocks, THREADS>>>(d_matr, d_vect, d_gpu_result);

    // reduce outputs of previous kernel blocks to single value
    singleBlockReduction<<<1, blocks>>>(d_gpu_result, blocks);

    cudaMemcpy(&gpu_result, d_gpu_result, sizeof(long int), cudaMemcpyDeviceToHost);

    cudaThreadSynchronize();
    cudaEventRecord(stop, 0);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&gpu_elapsed_time_ms, start, stop);
    printf("Time elapsed on GPU implementation with size of %d = %f ms.\n\n", DIM_SIZE, gpu_elapsed_time_ms);

    printf(gpu_result == cpu_result ? "OK" : "ERROR");
    printf("\n cpu_result: %d <-> gpu_result: %d", cpu_result, gpu_result);

    cudaFree(d_matr);
    cudaFree(d_vect);
    cudaFree(d_gpu_result);
    cudaFreeHost(matr);
    cudaFreeHost(vect);


    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
    {
        printf("CUDA Error: %s\n", cudaGetErrorString(err));
    }


    return 0;
}



Device Number: 0
  Device name: Tesla T4
  max Blocks Per MultiProcessor: 16
  max Threads Per MultiProcessor: 1024
  max Threads Per Block: 1024
  num SM: 40
  num bytes sharedMem Per Block: 49152
  num bytes sharedMem Per Multiprocessor: 65536
  Memory Clock Rate (KHz): 5001000
  Memory Bus Width (bits): 256
  Peak Memory Bandwidth (GB/s): 320.064000

Time elapsed on initialization with size 16384 = 5191.073000 ms

Time elapsed on naive CPU implementation with size 16384 = 966.719000 ms

Time elapsed on GPU implementation with size of 16384 = 31.521313 ms.

OK
 cpu_result: 396334905 <-> gpu_result: 396334905
